# Muster trace presentation

This notebook is deliberately downstream of the Go engine. It consumes JSON emitted by `muster replay --json`; it does not reproduce replay semantics.

Generate two Gate 2A runs from the repository root:

```bash
mkdir -p runs

go run . replay --scenario examples/gate2-early.yaml --controls examples/gate2-broad-correlated.yaml --json > runs/gate2-early-broad.json
go run . replay --scenario examples/gate2-between.yaml --controls examples/gate2-broad-correlated.yaml --json > runs/gate2-between-broad.json
```


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

RUN_DIR = Path("../runs")


def snake(name):
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1).lower()


def normalize(value):
    if isinstance(value, dict):
        return {snake(k): normalize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize(v) for v in value]
    return value


def load_replay(path):
    with Path(path).open() as f:
        doc = normalize(json.load(f))
    # CLI output wraps the engine result so the control-set id is retained.
    if "result" in doc:
        result = doc["result"]
        result["control_set_id"] = doc.get("control_set_id")
        return result
    return doc


## Timeline view

Each row is an incident event. `added` / `removed` are computed from the engine's before/after state snapshots, while `acted_controls` comes directly from the recorded control results.


In [ ]:
def timeline(run):
    rows = []
    for entry in run["trace"]:
        before = set(entry.get("before") or [])
        after = set(entry.get("after") or [])
        acted = [
            c["control_id"]
            for c in (entry.get("controls") or [])
            if c.get("acted")
        ]
        rows.append({
            "index": entry["index"],
            "event": entry["event_id"],
            "status": entry["status"],
            "added": ", ".join(sorted(after - before)),
            "removed": ", ".join(sorted(before - after)),
            "acted_controls": ", ".join(acted),
        })
    return pd.DataFrame(rows)


def control_trace(run):
    rows = []
    for entry in run["trace"]:
        for control in entry.get("controls") or []:
            rows.append({
                "event": entry["event_id"],
                "control": control["control_id"],
                "role": control.get("role", ""),
                "disposition": control["disposition"],
                "reason": control.get("reason", ""),
                "action": control.get("action", ""),
                "acted": control.get("acted", False),
            })
    return pd.DataFrame(rows)


In [ ]:
early_path = RUN_DIR / "gate2-early-broad.json"
between_path = RUN_DIR / "gate2-between-broad.json"

if early_path.exists() and between_path.exists():
    early = load_replay(early_path)
    between = load_replay(between_path)
    print("EARLY")
    display(timeline(early))
    print("BETWEEN")
    display(timeline(between))
else:
    print("Generate the two JSON runs shown at the top of the notebook first.")


## History-equivalence checkpoint

For Gate 2A, align the traces after `worker-code-execution`, `cluster-credential-theft`, and `worker-telemetry-impairment` have all been encountered, regardless of order. This exposes defensive-state differences without pretending the timelines have identical indices.


In [ ]:
def state_after_seen(run, event_ids):
    target = set(event_ids)
    seen = set()
    for entry in run["trace"]:
        seen.add(entry["event_id"])
        if target <= seen:
            return set(entry.get("after") or [])
    raise ValueError(f"trace never encountered all checkpoint events: {sorted(target)}")


def state_diff(a, b):
    return pd.DataFrame({
        "only_a": pd.Series(sorted(a - b), dtype="object"),
        "only_b": pd.Series(sorted(b - a), dtype="object"),
    })

checkpoint = {
    "worker-code-execution",
    "cluster-credential-theft",
    "worker-telemetry-impairment",
}

if early_path.exists() and between_path.exists():
    early_state = state_after_seen(early, checkpoint)
    between_state = state_after_seen(between, checkpoint)
    display(state_diff(early_state, between_state).rename(
        columns={"only_a": "EARLY only", "only_b": "BETWEEN only"}
    ))


## Side-by-side event comparison

Align by event id rather than trace index. This is useful when the same incident events occur in different orders.


In [ ]:
def compare_events(a, b, label_a="A", label_b="B"):
    def index(run):
        return {entry["event_id"]: entry for entry in run["trace"]}

    ai, bi = index(a), index(b)
    order = []
    for run in (a, b):
        for entry in run["trace"]:
            if entry["event_id"] not in order:
                order.append(entry["event_id"])

    rows = []
    for event_id in order:
        row = {"event": event_id}
        for label, item in ((label_a, ai.get(event_id)), (label_b, bi.get(event_id))):
            if item is None:
                row[f"{label}_status"] = "missing"
                row[f"{label}_acted"] = ""
                continue
            row[f"{label}_status"] = item["status"]
            row[f"{label}_acted"] = ", ".join(
                c["control_id"]
                for c in (item.get("controls") or [])
                if c.get("acted")
            )
        rows.append(row)
    return pd.DataFrame(rows)

if early_path.exists() and between_path.exists():
    display(compare_events(early, between, "EARLY", "BETWEEN"))


## Small experiment matrices

For paper tables, point a matrix at JSON outputs rather than hand-copying terminal outcomes.


In [ ]:
def adverse(path, fact="attacker:node-access"):
    run = load_replay(path)
    return fact in set(run.get("terminal_state") or [])


def outcome_matrix(paths, adverse_fact="attacker:node-access"):
    rows = []
    for row_name, columns in paths.items():
        row = {"architecture": row_name}
        for column_name, path in columns.items():
            path = Path(path)
            row[column_name] = (
                "NODE" if adverse(path, adverse_fact) else "secure"
            ) if path.exists() else "missing"
        rows.append(row)
    return pd.DataFrame(rows).set_index("architecture")

# Example:
# display(outcome_matrix({
#     "broad": {
#         "EARLY": RUN_DIR / "gate2-early-broad.json",
#         "BETWEEN": RUN_DIR / "gate2-between-broad.json",
#     }
# }))


## Presentation-layer rule

If this notebook ever needs to determine whether an event *should* apply, whether a control *should* act, or how an effect mutates state, stop and move that logic back into Go. The notebook is allowed to compare and present engine output—not reinterpret it.
